# 1 Exercise-1
## 1.1 Data Exploration
### Q.1.1.1

Examine the summary statistics (mean, standard deviation, min, max, quartiles) of
all 81 features and the target. Which features show the highest variation? Justify
why comparing raw standard deviations across these features can be misleading,
and propose a more suitable measure.

In [ ]:
from pathlib import Path
import pandas as pd
import zipfile
import urllib.request

def load_superconductivity_data():
    zip_path = Path("datasets/superconductivty+data.zip")
    if not zip_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://archive.ics.uci.edu/static/public/464/superconductivty+data.zip"
        urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(path="datasets/superconductivity")
    return pd.read_csv(Path("datasets/superconductivity/train.csv"))

data = load_superconductivity_data()

data.describe()

In [ ]:
features = data.drop(columns=['critical_temp'])
# if the 'material' formula column ended up in there too, drop that as well:
# features = data.drop(columns=['critical_temp', 'material'])

desc = features.describe().T

# Q: which features show the highest variation, by raw std?
top_std = desc.sort_values('std', ascending=False)
print(top_std[['mean', 'std']].head(10)) #I can add more, like min, max and etc., but to prove that raw std can be misleading, we will keep these two

bottom_std = desc.sort_values('std', ascending=True)
print(bottom_std[['mean', 'std']].head(10))


Sorting the features by `std` (from `describe()`) shows that all `*_Density` features
have the highest raw standard deviation. This is because density is measured in kg/m³, so its values sit
on a much larger numeric scale. This makes comparing raw standard deviations across features misleading, since a feature's raw std reflects its unit/scale as much as its actual variability.

A more suitable measure is the **coefficient of variation** (CV) (`std / mean`). Accord to Science direct CV is a statistic that is the ratio of the standard deviation to the mean expressed in percentage (https://www.sciencedirect.com/topics/engineering/coefficient-of-variation).
  Which is scale-free and lets us compare relative spread regardless of units. Here is an example of the use of CV.

In [ ]:
desc['CV_%'] = (desc['std'] / desc['mean']).abs() * 100   # coefficient of variation
top_cv = desc.sort_values('CV_%', ascending=False)
print(top_cv[['mean', 'std', 'CV_%']].head(10))

bottom_cv = desc.sort_values('CV_%', ascending=True)
print(top_cv[['mean', 'std', 'CV_%']].head(10))

Ranking by CV instead of raw std gives a completely different result. `wtd_gmean_ThermalConductivity`,
`wtd_range_FusionHeat`, and `wtd_gmean_FusionHeat` now on the top and all density related features do not stand out. This confirms that
Density's high raw std was mostly a scale effect, not genuinely higher relative variability.

## Q.1.1.2

Plot the distribution of critical_temp before and after a log transform. Based on
the shape of the distributions, discuss whether the raw or the transformed target
should be used for regression later in this assignment.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

y = data['critical_temp'] # raw target, right-skewed
y_log = np.log1p(y)       # to reduce the risk of log(0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left plot: distribution of the raw target
axes[0].hist(y, bins=50, color='red', edgecolor='black')
axes[0].set_title(f'Raw critical_temp (skew={y.skew():.2f})')  # skew() quantifies how asymmetric the distribution is

# Right plot: distribution after log transform
axes[1].hist(y_log, bins=50, color='yellow', edgecolor='black')
axes[1].set_title(f'log1p(critical_temp) (skew={y_log.skew():.2f})')

plt.tight_layout()  # prevents titles/labels from overlapping between the two subplots
plt.show()

For regression, the **log-transformed target is preferable**. The raw `critical_temp` is right-skewed (skew ≈ 0.86): most materials have a low
critical temperature, but a few have unusually high values, creating a long tail.

(A right-skewed distribution has a long tail toward high values; a left-skewed one has
a long tail toward low values. A skew of 0 means the distribution is symmetric.)

Since regression models typically minimize squared error, these few high-value
outliers would have an outsized effect on the fit compared to the bulk of the data.
After `log1p`, the distribution becomes close to symmetric (skew ≈ -0.32), which is
a better match for standard regression assumptions.

One practical note: predictions made on the log scale must be back-transformed with
`np.expm1()` before computing error metrics (RMSE, MAE), so results stay interpretable
in Kelvin.

## 1.2 Correlation Analysis
### Q1.2.1
Compute the correlation matrix of all features. Since a full 81 × 81 heatmap
is unreadable, plot a heatmap restricted to the 15 features most correlated with
critical_temp

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

target = 'critical_temp'

# 1. Compute the full correlation matrix
correlation_matrix = data.corr(numeric_only=True)

# 2. Get correlations with the target only, dropping the target's self-correlation (always 1.0)
target_correlation = correlation_matrix[target].drop(target)

# 3. Rank by absolute correlation (so strong negative correlations count too) and take top 15
top15 = target_correlation.abs().sort_values(ascending=False).head(15)
print(top15)

# 4. Build a small correlation matrix restricted to those 15 features + the target
top15_features = top15.index
subset_corr = data[list(top15_features) + [target]].corr()

# 5. Plot the heatmap
plt.figure(figsize=(15, 10))
sns.heatmap(subset_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation heatmap: 15 features most correlated with critical_temp')
plt.tight_layout()
plt.show()

The heatmap illustrates the correlation between the different features, and it ranges
from -1 to 1:

* +1 = perfect positive correlation (both go up together)
* -1 = perfect negative correlation (one goes up, the other goes down)
* 0 = no linear relationship

We chose to use a 15×15 subset because a full 81×81 correlation heatmap would contain
over 6,500 cells, which is unreadable. Restricting to the 15 features most correlated
with `critical_temp` keeps the heatmap readable while still highlighting the most
relevant relationships.

### Q1.2.2
Which variable has the strongest positive correlation with critical_temp? Which
variable has the strongest negative correlation?

In [ ]:
sorted_corr = correlation_matrix[target].sort_values(ascending=False)

print(sorted_corr.head(2))   # top row is critical_temp itself (1.00), but we ignore it
print(sorted_corr.tail(2))   # most negative correlations are at the bottom

The strongest positive correlation with `critical_temp` is `wtd_std_ThermalConductivity`
(r ≈ 0.72). The strongest negative correlation is `wtd_mean_Valence`
(r ≈ -0.63).

### Q1.2.3
Identify at least 5 pairs of features that are highly correlated with each other (ab-
solute correlation above 0.9), independent of their relationship with the target.
What problem does this create for a linear regression model, and how could it be
addressed?

In [ ]:
import numpy as np

features = data.drop(columns=[target])  # exclude the target, we only want feature-vs-feature correlations
corr = features.corr(numeric_only=True)

# corr is a square, symmetric matrix: corr[A][B] == corr[B][A], and the diagonal
# is always 1.00 (every feature correlates perfectly with itself).
# We only want each pair counted once, so we keep just the values ABOVE the
# diagonal (the "upper triangle") and throw away everything else.
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)  # k=1 skips the diagonal too
pairs = corr.where(mask).stack()  # keep only the masked values, drop the rest (NaNs)

high_corr = pairs[pairs.abs() > 0.9].sort_values(ascending=False)
print(f'Number of pairs with |corr| > 0.9: {len(high_corr)}')
print(high_corr.head(10))

When two features are highly correlated, they carry almost the same information.
Linear regression tries to assign each feature its own "weight" (coefficient) to show
how much it affects the target. But if two features move together, the model can't
tell which one is actually responsible for changes in critical_temp. It could give
credit to either one, or split it between them in a somewhat arbitrary way. This makes
the coefficients unstable: a tiny change in the data (or even just adding/removing a
few rows) can flip a coefficient from positive to negative, or make it swing wildly in
size, even though the model's overall predictions barely change. This makes the
coefficients unreliable to interpret (this is called **multicollinearity**).

### Q1.2.4
Using your correlation analysis, select one feature you expect to be a strong pre-
dictor of critical_temp and one you expect to be a weak predictor. Justify your
choice; you will use these two features in the next section.

In [ ]:
corr = data.corr(numeric_only=True)[target].drop(target)
weakest = corr.abs().sort_values().head(5)
print(weakest)

As a strong predictor, we select **`wtd_std_ThermalConductivity`** (r ≈ 0.72 with
`critical_temp`), the highest correlation found in our analysis, suggesting a clear
linear relationship with the target.

As a weak predictor, we select **`gmean_fie`** (r ≈ 0.03), which shows almost no
linear relationship with `critical_temp`, making it a useful contrast case.

## 1.3 Linear Regression

### Q1.3.1
Fit a simple linear regression model using gradient descent to predict critical_temp
using only the weak predictor selected in Q1.2.4. Standardize the feature before
running gradient descent and explain why this matters here.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

weak_feature = 'gmean_fie'   # from Q1.2.4

X = data[[weak_feature]].values
y = data[[target]].values

# split BEFORE scaling, so no information from the test set leaks into training
X_train, X_test, y_train_weak, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# fit the scaler on the training data only, then apply it to both sets
scaler_weak = StandardScaler()
X_train_scaled_weak = scaler_weak.fit_transform(X_train)
X_test_scaled_weak = scaler_weak.transform(X_test)

# add the bias column (x0 = 1) for the intercept term, same as the book does
X_train_b = np.c_[np.ones((len(X_train_scaled_weak), 1)), X_train_scaled_weak]

eta = 0.1          # learning rate
n_epochs = 1000
m = len(X_train_b)

np.random.seed(42)
theta_weak = np.random.randn(2, 1)   # [intercept, coefficient]

for epoch in range(n_epochs):
    gradients = 2 / m * X_train_b.T @ (X_train_b @ theta_weak - y_train_weak)
    theta_weak = theta_weak - eta * gradients

print("theta (intercept, coefficient):", theta_weak.ravel())

Without standardization, gradient descent diverges:

With a learning rate of 0.1, gradient descent didn't work on the raw feature. The
numbers blew up to nonsense within 50 steps. This happened because `gmean_fie` has
large raw values (roughly 375 to 1300), so each gradient descent step overshot by a
huge amount and kept getting worse instead of better. After standardizing the feature
(rescaling it to mean 0, standard deviation 1), the same learning rate worked fine and
the model converged normally.

Here is the code if we do not standardize the raw data:

In [ ]:
# 1. Check the raw range of the feature
print(X_train.min(), X_train.max())

# 2. Run gradient descent on the UNSCALED feature (no StandardScaler), same settings
X_train_b_raw = np.c_[np.ones((len(X_train), 1)), X_train]

eta = 0.1
n_epochs = 1000
m = len(X_train_b_raw)

np.random.seed(42)
theta_raw = np.random.randn(2, 1)

for epoch in range(n_epochs):
    gradients = 2 / m * X_train_b_raw.T @ (X_train_b_raw @ theta_raw - y_train)
    theta_raw = theta_raw - eta * gradients

print("theta on RAW (unscaled) feature:", theta_raw.ravel())

### Q1.3.2
Fit a simple linear regression model predicting critical_temp using only the strong
predictor selected in Q1.2.4.

In [ ]:
corr = data.corr(numeric_only=True)[target].drop(target)
strong = corr.abs().idxmax()   # idxmax() returns the index (feature name) with the highest value
print(strong)   # should print 'wtd_std_ThermalConductivity'

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = data[[strong]].values   # from Q1.2.4
y = data[[target]].values

# split BEFORE scaling, so no information from the test set leaks into training
X_train, X_test, y_train_strong, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# fit the scaler on the training data only, then apply it to both sets
scaler_strong = StandardScaler()
X_train_scaled_strong = scaler_strong.fit_transform(X_train)
X_test_scaled_strong = scaler_strong.transform(X_test)

# add the bias column (x0 = 1) for the intercept term, same as the book does
X_train_b = np.c_[np.ones((len(X_train_scaled_strong), 1)), X_train_scaled_strong]

eta = 0.1          # learning rate
n_epochs = 1000
m = len(X_train_b)

np.random.seed(42)
theta_strong = np.random.randn(2, 1)   # [intercept, coefficient]

for epoch in range(n_epochs):
    gradients = 2 / m * X_train_b.T @ (X_train_b @ theta_strong - y_train_strong)
    theta_strong = theta_strong - eta * gradients

print("theta (intercept, coefficient):", theta_strong.ravel())

### Q1.3.3
Report the regression coefficient and intercept and compare both models.

Both models have the same intercept (≈34.53), because standardizing the feature to
mean 0 makes the intercept equal to the mean of critical_temp in the training set.
This holds regardless of which feature is used.

The strong predictor has a large positive slope (+24.7): higher
`wtd_std_ThermalConductivity` is associated with meaningfully higher predicted
`critical_temp`. The weak predictor has a slope close to zero (-0.87), reflecting that
`gmean_fie` has almost no linear relationship with the target.

### Q1.3.4
Plot the regression line against the data points for both features. Does the regression line fit the data well in either case? Why or why not?

In [ ]:
import matplotlib.pyplot as plt

def plot_model(X_train_scaled, y_train, theta, feature_name, title, ax):
    ax.scatter(X_train_scaled, y_train, label='Data')  # default style, no alpha/markersize tweaks

    X_new = np.array([[X_train_scaled.min()], [X_train_scaled.max()]])
    X_new_b = np.c_[np.ones((2, 1)), X_new]
    y_predict = X_new_b @ theta

    ax.plot(X_new, y_predict, 'r-', label='Predicted critical_temp')
    ax.set_xlabel(feature_name)
    ax.set_ylabel('critical_temp')
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_model(X_train_scaled_weak, y_train_weak, theta_weak, 'line fits perfectly', 'Linreg (weak predictor)', axes[0])
plot_model(X_train_scaled_strong, y_train_strong, theta_strong, 'wtd_std_ThermalConductivity', 'Linreg (strong predictor)', axes[1])
plt.tight_layout()
plt.show()

From the graphs illustrated above, the line does not fit perfectly in either case.
For `gmean_fie`, the line is nearly flat. This is likely because the feature has almost
no real linear relationship with `critical_temp`. The `wtd_std_ThermalConductivity`
model shows a positive upward trend, but still has a lot of scatter around the line.
This can be because a single feature cannot fully predict `critical_temp`, when there
are 81 features available in total. It's a bit like shooting a tank with a BB gun.

## Train-Test Split

### Q1.4.1

Split the dataset into training (80%) and test (20%) sets in 5 different folds. Train the
simple linear regression model (using gradient descent, with standardized features) for
each split on the train-test data in each fold. Evaluate the model on the test set in each
fold using:
* Mean Squared Error (MSE)
* Root Mean Squared Error (RMSE)
* R2 score


In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

feature = strong   # or weak_feature — run this for whichever model the question wants

X = data[[feature]].values
y = data[[target]].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)   # 5 non-overlapping 80/20 folds
results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # same standardization + gradient descent code as Q1.3, just inside the loop now
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_b = np.c_[np.ones((len(X_train_scaled), 1)), X_train_scaled]
    X_test_b = np.c_[np.ones((len(X_test_scaled), 1)), X_test_scaled]

    eta, n_epochs = 0.1, 1000
    m = len(X_train_b)
    np.random.seed(42)
    theta = np.random.randn(2, 1)
    for epoch in range(n_epochs):
        gradients = 2 / m * X_train_b.T @ (X_train_b @ theta - y_train)
        theta = theta - eta * gradients

    y_pred = X_test_b @ theta
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    results.append({'fold': fold, 'MSE': mse, 'RMSE': rmse, 'R2': r2})

results_df = pd.DataFrame(results)
print(results_df)
print(results_df[['MSE', 'RMSE', 'R2']].describe())  # mean/std across folds, same as the book's pattern

import numpy as np

baseline_rmse = np.sqrt(np.mean((y_test - baseline_pred) ** 2))
print("Baseline RMSE (always guess the mean):", baseline_rmse)

print(results_df[['MSE', 'RMSE', 'R2']].agg(['mean', 'var'])) # Used in the last quest in Q1.4

The strong predictor explains about half the variance in `critical_temp` (R**2 ≈
0.51–0.53, meaning about 50% of the variance in critical_temp is explained by this
feature) across all folds, with very consistent results. On average, predictions are
off by about 23–24 K (RMSE — the typical size of the prediction error, in the same
units as critical_temp). In my opinion this is not a good model, but it makes sense
it is a 81 feature dataset.



### Q1.4.2
How well does the weak predictor alone predict critical_temp in each split?

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

feature = weak_feature   # or weak_feature — run this for whichever model the question wants

X = data[[feature]].values
y = data[[target]].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)   # 5 non-overlapping 80/20 folds
results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # same standardization + gradient descent code as Q1.3, just inside the loop now
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_b = np.c_[np.ones((len(X_train_scaled), 1)), X_train_scaled]
    X_test_b = np.c_[np.ones((len(X_test_scaled), 1)), X_test_scaled]

    eta, n_epochs = 0.1, 1000
    m = len(X_train_b)
    np.random.seed(42)
    theta = np.random.randn(2, 1)
    for epoch in range(n_epochs):
        gradients = 2 / m * X_train_b.T @ (X_train_b @ theta - y_train)
        theta = theta - eta * gradients

    y_pred = X_test_b @ theta
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    results.append({'fold': fold, 'MSE': mse, 'RMSE': rmse, 'R2': r2})

results_df = pd.DataFrame(results)
print(results_df)
print(results_df[['MSE', 'RMSE', 'R2']].describe())   # mean/std across folds, same as the book's pattern

baseline_rmse = np.sqrt(np.mean((y_test - baseline_pred) ** 2))
print("Baseline RMSE (always guess the mean):", baseline_rmse)

print(results_df[['MSE', 'RMSE', 'R2']].agg(['mean', 'var'])) # Used in the last quest in Q1.4

The weak predictor fails to predict `critical_temp` at all.
We can first observe the R2, where the value returned is close to 0. This tells us
that the model explains almost none of the variance and in two of the folds, R**2 is
negative (From what i understand the model is even worse than just guessing the mean).

This is confirmed by RMSE, which is nearly identical to the baseline RMSE (guessing
the mean every time, about 34.25 K) in every fold. Since the model's error is no
better than blindly guessing, this feature provides no real predictive value on its
own, consistent with its near zero correlation with `critical_temp` found in Q1.2.4.

### Q1.4.3
Do you think the model underfits? Why?
Yes, both models are underfitted and one can see that in the plots from 1.3.4, where the
regression lines clearly fail to capture the spread of the data. On can also confirm this
by comparing the training set and test performance: for both models, they are nearly identical.

* Strong: 0.520 train vs 0.520 test
* Weak: : 0.0006 train vs -0.0001 test

Since the models work so poorly, one can assume that the model might be underfit and that the
relation is more complex than using only using single features.


### Q1.4.4

Provide the mean and variance from the 5 different folds and comment on the
variation in performance across all 5 folds when using the strong predictor versus
the weak predictor.

This is taken from the last line from the previous code, the mean and variance across the
5 folds are:

**Strong:**
| | MSE | RMSE | R**2 |
|---|---|---|---|
| Mean | 563.18 | 23.73 | 0.5197 |
| Variance | 254.51 | 0.113 | 0.000067 |

**Weak:**
| | MSE | RMSE | R**2 |
|---|---|---|---|
| Mean | 1172.90 | 34.24 | -0.00013 |
| Variance | 1301.61 | 0.280 | 0.0000005 |

The table above shows that the Strong predicator outperforms the weaker one.

This can we see by comperaing the differernt values:

* lower mean MSE (563 vs 1173),
* lower mean RMSE (23.7 K vs 34.2 K)
* higher mean R**2 (0.52 vs 0)

The strong predictor's variance is also lower across all three metrics, meaning its
performance is better on average and also is more consistent in every fold.

## 1.5 Multiple Linear Regression
### Q1.5.1
Train a multiple linear regression model using all 81 features to predict critical_temp,
using the same splits as in the previous question. Evaluate the model on the test
set using MSE, RMSE, and R**2.

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

X = data.drop(columns=[target]).values   # using all 81 features
y = data[[target]].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)   # SAME splits as Q1.4
results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_b = np.c_[np.ones((len(X_train_scaled), 1)), X_train_scaled]
    X_test_b = np.c_[np.ones((len(X_test_scaled), 1)), X_test_scaled]

    eta, n_epochs = 0.03, 5000   # lower eta + more epochs than the single-feature model
    m = len(X_train_b)
    np.random.seed(42)
    theta = np.random.randn(X_train_b.shape[1], 1)   # 82 parameters now, not 2

    for epoch in range(n_epochs):
        gradients = 2 / m * X_train_b.T @ (X_train_b @ theta - y_train)
        theta = theta - eta * gradients

    y_pred = X_test_b @ theta
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    results.append({'fold': fold, 'MSE': mse, 'RMSE': rmse, 'R2': r2})

results_df = pd.DataFrame(results)
print(results_df)
print(results_df[['MSE', 'RMSE', 'R2']].agg(['mean', 'var']))

This model performed better than the single feature/linaer regression models,
but it was trickier to train. This is due to a learning rate
of 0.1 (used for the single-feature models) caused gradient descent to diverge (weights overflowed to NaN).
Here i just used trial and error, to find the best learning rate and landed on
lower value of 0.3 and with more epochs (5000) resolved this and produced stable
convergence.

### Q1.5.2
Compare the results of simple vs multiple regression in terms of MSE, RMSE, and
R2.

We can see that the R**2 in the multiple regression is between 0.716 and 0.737, which is
higher than either single-feature model (strong predictor: around 0.52; weak predictor: around0).
This proves the fact, (which was stated earlier) using all the 81 features are more captures much
more of the relationship with critical_temp than any single feature can alone.

### Q1.5.3
Provide comparison plots for multiple versus simple linear regression solved in the
previous exercise

Here have i choosen to stick to Cost vs. Iterastion since that was one of my biggest
struggels in this project.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Simple regression setup (strong predictor) ---
X_simple = data[[strong]].values
y = data[[target]].values

X_train_simple, X_test_simple, y_train_s, y_test_simple = train_test_split(
    X_simple, y, test_size=0.2, random_state=42)

scaler_s = StandardScaler()
X_train_s_scaled = scaler_s.fit_transform(X_train_simple)
X_train_s_b = np.c_[np.ones((len(X_train_s_scaled), 1)), X_train_s_scaled]

# --- Multiple regression setup (all 81 features) ---
X_multi = data.drop(columns=[target]).values

X_train_multi, X_test_multi, y_train_m, y_test_multi = train_test_split(
    X_multi, y, test_size=0.2, random_state=42)

scaler_m = StandardScaler()
X_train_m_scaled = scaler_m.fit_transform(X_train_multi)
X_train_m_b = np.c_[np.ones((len(X_train_m_scaled), 1)), X_train_m_scaled]

# --- Simple regression: track cost per epoch ---
np.random.seed(42)
theta_s = np.random.randn(2, 1)
cost_history_simple = []
m = len(X_train_s_b)
for epoch in range(1000):
    error = X_train_s_b @ theta_s - y_train_s
    cost_history_simple.append(np.mean(error ** 2))
    gradients = 2 / m * X_train_s_b.T @ error
    theta_s = theta_s - 0.1 * gradients

# --- Multiple regression: track cost per epoch ---
np.random.seed(42)
theta_m = np.random.randn(X_train_m_b.shape[1], 1)
cost_history_multi = []
m2 = len(X_train_m_b)
for epoch in range(5000):
    error = X_train_m_b @ theta_m - y_train_m
    cost_history_multi.append(np.mean(error ** 2))
    gradients = 2 / m2 * X_train_m_b.T @ error
    theta_m = theta_m - 0.03 * gradients

# --- Plot ---
plt.figure(figsize=(8, 5))
plt.plot(cost_history_simple, color='blue', label='Simple regression (strong predictor)')
plt.plot(cost_history_multi, color='yellow', label='Multiple regression (81 features)')
plt.xlim(0, 300)   # zoomed in, since both curves are flat well before 300 epochs
plt.xlabel('Iteration (epoch)')
plt.ylabel('Cost (MSE)')
plt.title('Cost vs Iteration: Simple vs Multiple Regression')
plt.legend()
plt.tight_layout()
plt.show()

### Q1.5.4
Which model performs better and why? Given the multicollinearity identified in
Q1.2.3, do you trust the individual coefficients of the multiple regression model?
Explain.

The multiple regression model performs better than the single-feature model. The reason
Behind it is that it has access to all the features.

Both models are show the classic
gradient descent shape, where the cost drop sharply and then flattens out. One thing to
note is that the multiple regression model takes noticeably longer to flatten out and
starts from a higher initial cost, since it has 81 parameters to adjust instead of 1.
